# File Input and Output

Files turn a short-lived process into a reusable tool. This notebook covers paths, encodings, open modes, reading, writing, appending, CSV parsing, and error handling.

## Learning goals

- Build portable paths with `pathlib`.
- Read a file as a whole and line by line.
- Write and append text without losing data.
- Use context managers to guarantee that files are closed.
- Parse CSV data with the standard library.
- Handle missing files and decoding errors explicitly.

## 1. Locating the data directory

The helper below works whether the notebook is launched from the session directory or from its `notebooks` directory. It searches the current directory and its parent for a `data` folder.

In [ ]:
from pathlib import Path


def find_data_directory():
    for candidate in (Path.cwd(), Path.cwd().parent):
        data_directory = candidate / "data"
        if data_directory.is_dir():
            return data_directory
    raise FileNotFoundError("Could not find the session data directory")


DATA_DIR = find_data_directory()
print(DATA_DIR)

In [ ]:
text_file = DATA_DIR / "text" / "hello_world.txt"
print(text_file)
print(text_file.exists())
print(text_file.resolve())

A relative path is resolved against the current working directory. Resolving the path before opening a file makes debugging much easier when a path is wrong.

## 2. Reading a whole file

`read_text` opens, reads, and closes the file in one call. It is convenient for small files.

In [ ]:
content = text_file.read_text(encoding="utf-8")
print(repr(content))
print(content)

For line-oriented processing, iterate over the file object. This streams the file and keeps memory usage low.

In [ ]:
with text_file.open(encoding="utf-8") as handle:
    for line_number, line in enumerate(handle, start=1):
        print(line_number, line.rstrip("\n"))

In [ ]:
with text_file.open(encoding="utf-8") as handle:
    lines = handle.readlines()

print(type(lines))
print(len(lines))
print(repr(lines[0]))

`readlines()` keeps the trailing newline on every line. That is useful when you want to preserve the exact text and inconvenient when you want to compare words.

## 3. Writing and appending

The next example writes to a temporary directory, so it leaves no generated files in the repository. Mode `"w"` truncates an existing file; mode `"a"` appends.

In [ ]:
from tempfile import TemporaryDirectory

with TemporaryDirectory() as directory:
    output = Path(directory) / "notes.txt"
    with output.open("w", encoding="utf-8", newline="\n") as handle:
        handle.write("first line\n")
        handle.write("second line\n")

    with output.open("a", encoding="utf-8", newline="\n") as handle:
        handle.write("third line\n")

    print(output.read_text(encoding="utf-8"))

The `with` statement closes the file even if an exception occurs. Without it, data may not be flushed and file descriptors may leak. `write` does not add a newline automatically; include `\n` explicitly.

## 4. Parsing a CSV file manually

Manual parsing is useful for learning. Always strip the newline before splitting the last field.

In [ ]:
area_price_path = DATA_DIR / "area_price.csv"
with area_price_path.open(encoding="utf-8") as handle:
    header = next(handle).strip().split(",")
    rows = []
    for line in handle:
        fields = line.strip().split(",")
        rows.append(dict(zip(header, fields, strict=True)))

print(header)
print(rows)

## 5. Using the `csv` module

The `csv` module handles quoting, escaped delimiters, and embedded newlines. Open the file with `newline=""` so that the module controls newline translation.

In [ ]:
import csv

iris_path = DATA_DIR / "iris.csv"
with iris_path.open(encoding="utf-8", newline="") as handle:
    reader = csv.DictReader(handle)
    records = list(reader)

print(reader.fieldnames)
print(records[0])
print(len(records))

## 6. Computing statistics

Separate parsing from aggregation. The parsing loop converts each field to the correct type and rejects malformed records; the aggregation step then works with clean numbers.

In [ ]:
petal_path = DATA_DIR / "iris_petal.csv"
lengths = []
with petal_path.open(encoding="utf-8", newline="") as handle:
    reader = csv.DictReader(handle)
    for line_number, row in enumerate(reader, start=2):
        try:
            lengths.append(float(row["PetalLength"]))
        except (TypeError, ValueError) as error:
            raise ValueError(f"invalid PetalLength on line {line_number}") from error

mean_length = sum(lengths) / len(lengths)
print(f"count={len(lengths)}")
print(f"mean={mean_length:.4f}")
print(f"min={min(lengths):.2f}")
print(f"max={max(lengths):.2f}")

## 7. Handling missing files

Catch the specific exception you can handle, and include the path in the message. Avoid a bare `except`, which hides programming errors and interrupts.

In [ ]:
missing_path = DATA_DIR / "text" / "missing.txt"
try:
    missing_path.read_text(encoding="utf-8")
except FileNotFoundError:
    print(f"File not found: {missing_path}")
except PermissionError:
    print(f"Permission denied: {missing_path}")
except UnicodeDecodeError as error:
    print(f"Could not decode {missing_path}: {error}")

## 8. A small pipeline

The pipeline below loads student records, validates each field, produces a sorted report, and writes it to a temporary file. The functions are small enough to test independently.

In [ ]:
def load_students(path):
    students = []
    with path.open(encoding="utf-8", newline="") as handle:
        reader = csv.DictReader(handle)
        for line_number, row in enumerate(reader, start=2):
            identifier = row["student_id"].strip()
            name = row["name"].strip()
            date_of_birth = row["date_of_birth"].strip()
            if not identifier or not name or not date_of_birth:
                raise ValueError(f"missing field on line {line_number}")
            students.append(
                {
                    "student_id": identifier,
                    "name": name,
                    "date_of_birth": date_of_birth,
                }
            )
    return students


def build_report(students):
    lines = [f"student count: {len(students)}"]
    for student in sorted(students, key=lambda item: item["name"]):
        lines.append(f"{student['student_id']}: {student['name']}")
    return "\n".join(lines) + "\n"


students = load_students(DATA_DIR / "text" / "students.csv")
report = build_report(students)
print(report)

In [ ]:
with TemporaryDirectory() as directory:
    output = Path(directory) / "report.txt"
    output.write_text(report, encoding="utf-8")
    print(output.read_text(encoding="utf-8"))

## 9. Common file errors

| Error | Cause | Fix |
| :--- | :--- | :--- |
| `FileNotFoundError` | Wrong path or missing file | Print the resolved path and check `exists()`. |
| `PermissionError` | No read or write permission | Choose a writable location. |
| `IsADirectoryError` | Opening a directory as a file | Supply a file path. |
| `UnicodeDecodeError` | Wrong encoding | Read with the correct encoding, usually UTF-8. |
| Empty file after writing | Data not flushed or file opened with `"w"` twice | Use one `with` block and write before closing. |

## Exercises

1. Read `area_price.csv` and print the average price.
2. Read `iris.csv` and count the records for each species.
3. Read `iris_petal.csv` and compute the mean petal width for each label.
4. Write a function that copies a text file and normalises line endings to `\n`.
5. Extend the student report with a count of names that start with each letter.
6. Catch `UnicodeDecodeError`, retry with `latin-1`, and explain why the result may differ.